In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import sys
import os

# On remonte d'un cran si on est dans un sous-dossier, 
# ou on utilise le chemin absolu de ton dossier utilisateur
project_root = os.path.abspath(os.path.expanduser("~/MedReport-AI-Diagnostics"))

if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Racine du projet ajoutée : {project_root}")

# Vérification : est-ce que le dossier src est bien là ?
if os.path.exists(os.path.join(project_root, "src")):
    print("✅ Dossier 'src' détecté !")
else:
    print("❌ Dossier 'src' introuvable. Vérifie le nom du dossier racine.")
# Importation de tes modules personnalisés
from src.data_loader import MedReportDataset
from src.model import get_model

# Configuration du matériel (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"L'entraînement se déroulera sur : {device}")

Racine du projet ajoutée : /home/lethycia/MedReport-AI-Diagnostics
✅ Dossier 'src' détecté !
L'entraînement se déroulera sur : cpu


In [2]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# Détection automatique de la racine du projet
project_root = os.path.abspath(os.path.expanduser("~/MedReport-AI-Diagnostics"))
if project_root not in sys.path:
    sys.path.append(project_root)

# Importation de tes modules (on vérifie s'ils existent d'abord)
try:
    from src.data_loader import MedReportDataset
    from src.model import get_model
    print("✅ Modules 'src' chargés avec succès.")
except ImportError:
    print("❌ Erreur : Dossier 'src' ou fichiers .py introuvables dans", project_root)

# Configuration Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"L'entraînement se déroulera sur : {device}")

✅ Modules 'src' chargés avec succès.
L'entraînement se déroulera sur : cpu


In [3]:
# Chemins absolus
csv_path = os.path.join(project_root, 'data', 'Data_Entry_2017.csv')
img_dir = os.path.join(project_root, 'data', 'images')

# Chargement du CSV
if not os.path.exists(csv_path):
    print(f"❌ Fichier CSV non trouvé à : {csv_path}")
else:
    df = pd.read_csv(csv_path)
    
    # Filtrage : On ne garde que les images présentes dans tes 2 Go
    df = df[df['Image Index'].apply(lambda x: os.path.exists(os.path.join(img_dir, x)))]
    
    # Création des colonnes de labels
    labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 
              'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 
              'Fibrosis', 'Pleural_Thickening', 'Hernia']

    for l in labels:
        df[l] = df['Finding Labels'].map(lambda x: 1 if l in x else 0)

    # 1. On splitte d'abord pour mettre de côté le Test Set (20%)
    train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    
    # 2. On splitte le reste en Train et Validation (80% de 80% = 64% total / 20% de 80% = 16% total)
    train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)
    
    print(f"Images d'entraînement : {len(train_df)}")
    print(f"Images de validation  : {len(val_df)}")
    print(f"Images de test (final) : {len(test_df)}")

    


Images d'entraînement : 3199
Images de validation  : 800
Images de test (final) : 1000


In [4]:
# Transformations (Standard ImageNet)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Création des objets Dataset et DataLoader
train_dataset = MedReportDataset(train_df, img_dir, transform=data_transforms)
val_dataset = MedReportDataset(val_df, img_dir, transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("✅ DataLoaders créés.")

✅ DataLoaders créés.


In [5]:
# Sauvegarde facultative du test_df pour demain
test_df.to_csv(os.path.join(project_root, 'data', 'test_subset.csv'), index=False)

In [6]:
# Initialisation
model = get_model(num_classes=14).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 5 # Tu peux augmenter si ton ThinkPad suit !
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
model_save_path = os.path.join(project_root, 'models', 'best_medreport_model.pth')

os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

print("🚀 Début de l'entraînement...")

for epoch in range(EPOCHS):
    # --- ENTRAÎNEMENT ---
    model.train()
    running_train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * images.size(0)
    
    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # --- VALIDATION ---
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_val_loss += loss.item() * images.size(0)
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    
    # Historique pour les graphiques
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    # Sauvegarde si record battu
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), model_save_path)
        print(f"⭐ Score amélioré ! Modèle enregistré sous : {model_save_path}")

print("✅ Entraînement terminé.")

🚀 Début de l'entraînement...


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Perte Entraînement (Train)')
plt.plot(history['val_loss'], label='Perte Validation (Val)')
plt.title('Performance du modèle MedReport-AI')
plt.xlabel('Époques')
plt.ylabel('BCE Loss')
plt.legend()
plt.grid(True)
plt.show()